In [39]:
from langgraph.graph import StateGraph, START,END 
from typing import TypedDict
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("MODEL_NAME")

In [40]:
from  langchain_groq import ChatGroq
model=ChatGroq(
    api_key=api_key,
    model_name=model_name,
    temperature=0.0)

In [57]:
class BatsmanState(TypedDict):
    runs : int
    balls : int
    fours : int
    sixs : int

    sr : float
    bpb: float
    boundary_percent : float


In [58]:
def calculate_sr(state:BatsmanState):
    sr = (state['runs']/state['balls'])*100
    
    return {'sr':sr}

In [59]:
def calculate_bpb(state:BatsmanState):
    bpb=state['balls']/(state['fours'] + state['sixs'])

    return {"bpb":bpb}

In [60]:
def calculate_boundary_percent(state: BatsmanState):
    
    boundary_percent=(((state['fours']*4)+(state['sixs']*6))/state['runs'])*100
    
    return {'boundary_percent':boundary_percent}

In [61]:
def summary(state:BatsmanState):
    summary=f"""
Strike Rate - {state['sr']}\n
Balls per boundry - {state['bpb']}\n
Boundary percent - {state['boundary_percent']}
"""
    
    return {'summary': summary}

In [63]:
graph=StateGraph(BatsmanState)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary',summary)

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')
graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')

graph.add_edge('summary',END)

workflow=graph.compile()


In [64]:
init_state={
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixs': 4,
}
workflow.invoke(init_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixs': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0}